# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [2]:
url = "http://books.toscrape.com/catalogue/page-1.html"

response = requests.get(url)
print("Status Code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

response.raise_for_status()

Status Code: 200
Content-Type: text/html


In [3]:
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

first_book = soup.find("article", class_="product_pod")
display(first_book)

<article class="product_pod">
<div class="image_container">
<a href="a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
</div>
<p class="star-rating Three">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
<div class="product_price">
<p class="price_color">Â£51.77</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>

In [4]:
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

book_blocks = soup.find_all("article", class_="product_pod")
print("Books found on page:", len(book_blocks))

title = first_book.find("h3").find("a").get("title")
price_str = first_book.find("p", class_="price_color").get_text(strip=True)
price = float(re.sub(r"[^\d.]", "", price_str))

rating_class = first_book.find("p", class_="star-rating").get("class")[1]
rating = rating_map.get(rating_class, 0)

print(f"Title: {title}")
print(f"Price: {price}")
print(f"Rating: {rating}")

Books found on page: 20
Title: A Light in the Attic
Price: 51.77
Rating: 3


In [6]:
from urllib.parse import urljoin

relative_url = first_book.find("h3").find("a").get("href")
base_url = "http://books.toscrape.com/catalogue/"
book_url = urljoin(base_url, relative_url)

detail_response = requests.get(book_url)
detail_response.raise_for_status()

detail_soup = BeautifulSoup(detail_response.text, "html.parser")

availability = detail_soup.find("p", class_="availability").get_text(strip=True)

desc_tag = detail_soup.find("div", id="product_description")
description = desc_tag.find_next_sibling("p").get_text(strip=True) if desc_tag and desc_tag.find_next_sibling("p") else "No description"

table = detail_soup.find("table", class_="table-striped")
upc = table.find("th", string="UPC").find_next_sibling("td").get_text(strip=True) if table else "Unknown"

breadcrumb = detail_soup.find("ul", class_="breadcrumb")
genre = breadcrumb.find_all("li")[2].get_text(strip=True) if breadcrumb and len(breadcrumb.find_all("li")) >= 3 else "Unknown"

print(f"UPC: {upc}")
print(f"Genre: {genre}")
print(f"Availability: {availability}")

UPC: a897fe39b1053632
Genre: Poetry
Availability: In stock (22 available)


In [7]:
next_element = soup.find("li", class_="next")

if next_element is not None:
    next_href = next_element.find("a").get("href")
    next_url = urljoin(url, next_href)
    print("Next page URL:", next_url)
else:
    print("No more pages.")

Next page URL: http://books.toscrape.com/catalogue/page-2.html


In [10]:
def scrape_books(min_rating=4.0, max_price=20.0):
    base_url = "http://books.toscrape.com/catalogue/"
    current_url = "http://books.toscrape.com/catalogue/page-1.html"
    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    
    all_records = []
    
    while True:
        response = requests.get(current_url)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, "html.parser")
        book_blocks = soup.find_all("article", class_="product_pod")
        
        for block in book_blocks:

            title = block.find("h3").find("a").get("title")
            price_str = block.find("p", class_="price_color").get_text(strip=True)
            price = float(re.sub(r"[^\d.]", "", price_str))
            
            rating_class = block.find("p", class_="star-rating").get("class")[1]
            rating = rating_map.get(rating_class, 0)
            

            if rating < min_rating or price > max_price:
                continue
                

            relative_url = block.find("h3").find("a").get("href")
            book_url = urljoin(base_url, relative_url)
            
            detail_response = requests.get(book_url)
            if detail_response.status_code != 200:
                continue
            detail_soup = BeautifulSoup(detail_response.text, "html.parser")
            

            availability_tag = detail_soup.find("p", class_="availability")
            availability = availability_tag.get_text(strip=True) if availability_tag else "Unknown"
            
            desc_tag = detail_soup.find("div", id="product_description")
            description = desc_tag.find_next_sibling("p").get_text(strip=True) if desc_tag and desc_tag.find_next_sibling("p") else "No description"
            
            table = detail_soup.find("table", class_="table-striped")
            upc = "Unknown"
            if table:
                upc_row = table.find("th", string="UPC")
                if upc_row:
                    upc = upc_row.find_next_sibling("td").get_text(strip=True)
            
            breadcrumb = detail_soup.find("ul", class_="breadcrumb")
            genre = "Unknown"
            if breadcrumb:
                items = breadcrumb.find_all("li")
                if len(items) >= 3:
                    genre = items[2].get_text(strip=True)
            

            all_records.append({
                "UPC": upc,
                "Title": title,
                "Price (£)": price,
                "Rating": float(rating),
                "Genre": genre,
                "Availability": availability,
                "Description": description
            })
            

        next_element = soup.find("li", class_="next")
        if next_element is None:
            break
            
        current_url = urljoin(current_url, next_element.find("a").get("href"))
        

    df_books = pd.DataFrame(all_records)
    return df_books

In [11]:
filtered_books_df = scrape_books(min_rating=4.0, max_price=20.0)
display(filtered_books_df.head())
print("Total rows matching criteria:", len(filtered_books_df))

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5.0,Young Adult,In stock (19 available),Aaron Ledbetterâs future had been planned ou...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5.0,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5.0,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4.0,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4.0,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."


Total rows matching criteria: 75
